# Labeling Script also Providing Defnitions and Examples  

## TOC 

- [Setup](#setup)
- [Load, Partition, and Sample Tweets/Batches for Labeling](#load_tweets)
- [Define the different prompts](#define_prompts)
- [Run the Labeling Tasks (Single)](#run_labeling_single)
- [Run the Labeling Tasks (Single) + Confidence](#run_labeling_single_conf)
- [Run the Labeling Tasks (Batch)](#run_labeling_batch)
- [Run the Labeling Tasks (Batch) + Confidence](#run_labeling_batch_confidence)

#### Open ToDo's
- Clean tweets from nbsps etc.  
- rewrite data frame creation and data collection into separate functions  

- Update Collection prompt s.t. there are now unneccessary ' after splitting
 --> batch conf still contains some cases without proper scores

Update to previous versions: 

-separation of task by system message (prompt and def) and user message (tweet)

In [1]:
# General Setup and Git
# cd to path of current script
import os
import sys
os.chdir(sys.path[0])

! git pull

Already up to date.


In [2]:
# Git codes
## see git status
! git status # see git status

# Ask the user for a commit message
commit_message = input("Enter commit message: ")

## Git add the current stuff to commit and set commit message
! git add .
! git commit -m "$commit_message"

## Push commits to remote
! git push

On branch master
Your branch is up to date with 'origin/master'.

Changes not staged for commit:
  (use "git add/rm <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	deleted:    .ipynb_checkpoints/3c_GPT_Label_Certainty-checkpoint.ipynb
	modified:   .jupyter/desktop-workspaces/default-37a8.jupyterlab-workspace
	modified:   1a_GPT_Labeling.ipynb
	deleted:    3c_GPT_Label_Certainty.ipynb

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.ipynb_checkpoints/1d_Separate_Labeling_Definition_FewShot_refined_sep-checkpoint.ipynb
	.ipynb_checkpoints/2c_GPT_Label_Certainty-checkpoint.ipynb
	1d_Separate_Labeling_Definition_FewShot_refined_sep.ipynb
	2c_GPT_Label_Certainty.ipynb

no changes added to commit (use "git add" and/or "git commit -a")


Enter commit message:  separate Labeling


[master 09d0bda] separate Labeling
 6 files changed, 7724 insertions(+), 3 deletions(-)
 create mode 100644 .ipynb_checkpoints/1d_Separate_Labeling_Definition_FewShot_refined_sep-checkpoint.ipynb
 rename .ipynb_checkpoints/{3c_GPT_Label_Certainty-checkpoint.ipynb => 2c_GPT_Label_Certainty-checkpoint.ipynb} (100%)
 rewrite .jupyter/desktop-workspaces/default-37a8.jupyterlab-workspace (89%)
 create mode 100644 1d_Separate_Labeling_Definition_FewShot_refined_sep.ipynb
 rename 3c_GPT_Label_Certainty.ipynb => 2c_GPT_Label_Certainty.ipynb (100%)
Enumerating objects: 13, done.
Counting objects: 100% (13/13), done.
Delta compression using up to 8 threads
Compressing objects: 100% (6/6), done.
Writing objects: 100% (7/7), 2.06 KiB | 2.06 MiB/s, done.
Total 7 (delta 4), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (4/4), completed with 4 local objects.
To github.com:T-Reiter/master-thesis.git
   856ec9b..09d0bda  master -> master


<a id="setup"></a>
## SANDBOX & SETUP

In [79]:
# Load API key from gitignored api_key module 
from utils.api_key import API_KEY as API_KEY

In [83]:
# base Request
# test capabilties
from openai import OpenAI

# Setup open AI client using api key
client = OpenAI(
  api_key=API_KEY
)

# Test prompt
return_number = "Select a letter from the alphabet! Only return the letter"

N_TOP_LOGPROBS = 2

# Test completion
resp_full = client.chat.completions.create(
            model="gpt-4o",
             messages=[{"role": "user", 
                       "content": return_number
                       }],
            temperature=2,
            logprobs=True,
            top_logprobs= N_TOP_LOGPROBS,
            store=True,
        )

print("Response: ", resp_full.choices[0].message.content);
print('--------------------------------------------')
print("Response: ", resp_full.choices[0].logprobs.content);
print('--------------------------------------------')
print("Response: ", resp_full.choices[0].logprobs.content[0].top_logprobs);

resp_full

Response:  Q
--------------------------------------------
Response:  [ChatCompletionTokenLogprob(token='Q', bytes=[81], logprob=-5.296432018280029, top_logprobs=[TopLogprob(token='K', bytes=[75], logprob=-0.7964320778846741), TopLogprob(token='G', bytes=[71], logprob=-1.1714320182800293)])]
--------------------------------------------
Response:  [TopLogprob(token='K', bytes=[75], logprob=-0.7964320778846741), TopLogprob(token='G', bytes=[71], logprob=-1.1714320182800293)]


ChatCompletion(id='chatcmpl-BlyZQrD9C7wIKUQ3Q0fUMcuuL2v75', choices=[Choice(finish_reason='stop', index=0, logprobs=ChoiceLogprobs(content=[ChatCompletionTokenLogprob(token='Q', bytes=[81], logprob=-5.296432018280029, top_logprobs=[TopLogprob(token='K', bytes=[75], logprob=-0.7964320778846741), TopLogprob(token='G', bytes=[71], logprob=-1.1714320182800293)])], refusal=None), message=ChatCompletionMessage(content='Q', refusal=None, role='assistant', audio=None, function_call=None, tool_calls=None, annotations=[]))], created=1750773948, model='gpt-4o-2024-08-06', object='chat.completion', service_tier='default', system_fingerprint='fp_a288987b44', usage=CompletionUsage(completion_tokens=1, prompt_tokens=18, total_tokens=19, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))

## Logprob Investigation

Insights:   

The Response Object and what it contains: 
message.content: contains the actual model output --> i.e., the LLM response 






This is how we usually extract the response: 
label = resp.choices[0].message.content.strip() --> this response is actually affected by temperature

This is how we extract the logprobs for the first token: 
top_k_logprobs_first = resp_full.choices[0].logprobs.content[0].top_logprobs  
        --> get the first element from choices (unclear what this means)  
        --> get the logprobs   
        --> use the first element in the logprobs  [0]  
        --> top_logprobs --> this contains the top 5 tokens with their associated logprob  
        --> Theoretically these top 5 tokens not even have to contain the actual response (e.g. 2 top_logprobs for returning a random letter)  
        --> the other (first) element included ChatCompletionTokenLogprob is just again the values for the selected element  
        --> this element is the same as the response by the LLM (if there is only one token)  
        --> However this reponse does not necessarily have to be the token with the highest logprob   
        

**The Response output is not just a Function of the Logprobs but rather of the random selection out of the probability distribution the logprobs provide**  
**The logprobs themselves are not determninistic but change in every API Call**   
(basically we would assume that they are but different reasons are proposed why they are not (see below)  
--> at least, it seems that the general tendency remains rather stable  
--> temperature scales the logprob probability distribution (0 --> no further scaling, higher termpature --> more uniform)  
--> accordingly, temperature is relevant for the sampling process but nothing else

However, the logprobs themselves are not deterministic. One possible reason for this is proposed in the following:  
"There are other sources of randomness which are very difficult to eliminate, particularly the extreme parallelism of the transformer architecture."
"That’s the “efficency” of the gpt-4-turbo and new gpt-3.5-turbo-0125 for you. We don’t know why what should be the same calcuations yield different output from the AI."  
--> indeed, in the 40-mini model this issue seems to be bigger than in the other normal 4o
--> in the latter, at least the overall order of the top_p tokens does not change tooo often (although the logprobs slightly change from time to time)

--> Accordingly, temperature should not affect the actual token prediction distribution output but affect the token that is sampled in the end 
temperature 0 would always select the token with the highest probability and larger temperatures would tweak the distribution closer to a random distribution